<h3> Reading documents in the text format, and embedding page by page. </h3>

<h5> Imports </h5>

In [ ]:
import os
import json
import glob
import fitz 

from azure.identity import DefaultAzureCredential
from openai import AzureOpenAI

import numpy as np
from paddleocr import PaddleOCR

from dotenv import load_dotenv

load_dotenv()

<h5> Experimenting with PaddleOCR, drawback is that it is slow with documents with a high number of pages </h5>

In [ ]:
def ocr_document(pdf_path: str, dpi: int = 300, lang: str = "en", conf_min: float = 0.5, native_min_chars: int = 40):
    """
    OCR a single PDF document (path).
    - Extracts native text if available.
    - Falls back to PaddleOCR for scanned pages.
    Returns: list of dicts [{page, method, text}]
    """
    # Initialize PaddleOCR with correct parameters for v3.x
    ocr = PaddleOCR(use_textline_orientation=True, lang=lang)

    def page_to_image(page):
        zoom = dpi / 72
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
        if pix.n == 4:
            img = img[:, :, :3]
        return img

    doc = fitz.open(pdf_path)
    pages = []

    for i, page in enumerate(doc, start=1):
        native_text = (page.get_text("text") or "").strip()

        if len(native_text) >= native_min_chars:
            text = native_text
            method = "native"
        else:
            img = page_to_image(page)
            result = ocr.predict(img)
            lines = []
            if result and result[0]:
                for line in result[0]:
                    if line and len(line) >= 2:
                        # Structure: [[bbox], (text, confidence)]
                        txt, conf = line[1][0], line[1][1]
                        if conf >= conf_min:
                            lines.append(txt.strip())
            text = "\n".join(lines).strip()
            method = "ocr"

        pages.append({"page": i, "method": method, "text": text})

    doc.close()
    return pages

<h5> Since we just need a text, plain pdf to text should do the job. </h5>

In [ ]:
def extract_pdf_text_simple(pdf_path: str):
    """
    Extract text from PDF using native text extraction only.
    Returns: list of dicts [{page, text}]
    """
    doc = fitz.open(pdf_path)
    pages = []
    
    for i, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()
        pages.append({
            "page": i,
            "text": text,
            "method": "mypdf"
        })
    
    doc.close()
    return pages

<h5> Getting the embedding with the normalization is required because

In [ ]:
azure_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION")
)

def normalize(v):
    """ Normalize a vector to unit length """
    v = np.array(v, dtype="float32")
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v


def get_text_embedding(text, normalizeEmbedding=True, model="text-embedding-3-large"):
    """ Get text embedding from Azure OpenAI and optionally normalize it """
    emb = azure_client.embeddings.create(input=[text], model=model).data[0].embedding
    if normalizeEmbedding:
        emb = normalize(emb)
    return emb

In [ ]:
all_results = []

pdf_files = glob.glob("data/*.pdf")

print(f"Found {len(pdf_files)} PDF files")

for pdf_path in pdf_files:
    try:
        print(f"Processing: {pdf_path}: {pdf_files.index(pdf_path)+1}/{len(pdf_files)}")
        pages = extract_pdf_text_simple(pdf_path)
        all_results.append({
            "file": pdf_path,
            "pages": pages
        })
        print(f"Extracted {len(pages)} pages")
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")

print(f"Total files processed: {len(all_results)}")

<h5> Retrieve the embedding for each page of each document. </h5>

In [ ]:
for doc_result in all_results:
    file_name = doc_result["file"]
    pages = doc_result["pages"]
    
    print(f"Processing embeddings for: {file_name}")
    
    page_embeddings = []
    
    for page_data in pages:
        text = page_data["text"]
        
        if text.strip():
            try:
                embedding = get_text_embedding(text, normalizeEmbedding=True)
                page_embeddings.append(embedding)
                print(f"Page {page_data['page']}: embedding calculated ({len(embedding)} dims)")
            except Exception as e:
                print(f"Page {page_data['page']}: error - {e}")
        else:
            print(f"Page {page_data['page']}: empty text, skipping")
    
    doc_result["embeddings"] = page_embeddings

In [ ]:
# Create embeddings folder
embeddings_dir = os.path.join("data_output", "embeddings")
os.makedirs(embeddings_dir, exist_ok=True)

# Save each document to a separate file
for idx, doc in enumerate(all_results):
    # Create a filename based on the original PDF name
    original_filename = os.path.basename(doc["file"])
    base_name = os.path.splitext(original_filename)[0]
    output_filename = f"{base_name}_embeddings.json"
    output_path = os.path.join(embeddings_dir, output_filename)
    
    # Convert numpy arrays to lists for JSON serialization
    doc_to_save = doc.copy()
    if "embeddings" in doc_to_save:
        doc_to_save["embeddings"] = [emb.tolist() if isinstance(emb, np.ndarray) else emb 
                                      for emb in doc_to_save["embeddings"]]
    
    # Save individual document
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(doc_to_save, f, indent=2, ensure_ascii=False)
    
    print(f"Saved: {output_filename}")

print(f"\nTotal files saved: {len(all_results)}")
print(f"Saved to folder: {embeddings_dir}")

<h5> Store embeddings, extracted text from pages to a json file that corresponds to the original pdf name </h5>

In [ ]:
import json
import glob

# Read all JSON files from embeddings folder
embeddings_dir = os.path.join("data_output", "embeddings")
embeddings_files = glob.glob(os.path.join(embeddings_dir, "*.json"))

all_results = []

print(f"Found {len(embeddings_files)} JSON files in {embeddings_dir}")

for json_file in embeddings_files:
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            doc_data = json.load(f)
            all_results.append(doc_data)
        print(f"Loaded: {os.path.basename(json_file)}")
    except Exception as e:
        print(f"Error loading {json_file}: {e}")

print(f"\nTotal documents loaded: {len(all_results)}")